# Evaluation Review

This notebook lets you inspect the results of an evaluation run **after** running `make run-eval-uv`.

Each run writes a directory under `evals/output/`:

`evals/output/eval_run_<timestamp>_<ontology|no_ontology>/`

containing four files:

- `agent_response.csv` — one row per question, with the agent's answer and the judge scores
  (`rouge_f1_score`, `factual_correctness_f1_score`, `answer_relevancy_score`).
- `metadata.csv` — one row per question, with the generated Cypher, the tools the agent
  called, message/LLM/tool-call counts and response time.
- `failed_response.csv` — questions that errored before producing an answer (often empty).
- `report.txt` — the same per-question report this notebook prints, generated by the eval.

Start Jupyter from the **repo root** so the relative `evals/output/` path and the
`evaluation` package imports below work.

In [ ]:
import os
import sys
from ast import literal_eval

import pandas as pd

# Make the repo root importable so we can reuse the eval's own helpers
# (no-op when the notebook is already running from the repo root).
sys.path.insert(0, os.getcwd())

from evaluation.components.generate_report import (
    EVALS_PATH,
    create_report_str_from_row,
    get_most_recent_eval_run_directory,
)

## 1. Pick a run directory

By default we review the **most recent** run, using the same helper the report generator
uses. To review a different run (e.g. to compare the `ontology` and `no_ontology`
conditions), set `RUN_DIRECTORY` to its directory name.

In [ ]:
# Default: most recent run. Override with e.g.
# RUN_DIRECTORY = "eval_run_2026-11-11_16-09-01_ontology"
RUN_DIRECTORY = None

if RUN_DIRECTORY is None:
    RUN_DIRECTORY = get_most_recent_eval_run_directory()

run_path = os.path.join(EVALS_PATH, RUN_DIRECTORY)
print(f"Reviewing run: {run_path}")

## 2. Load the run's CSVs

`metadata.csv` stores `generated_cypher` and `called_tools` as Python-literal strings, so we
pass `ast.literal_eval` as a converter to get real lists back. (`available_tools` is a
stringified `repr` of tool objects and is left as-is.)

The `error` column exists in both `agent_response.csv` and `metadata.csv`; we rename each
to `agent_response_error` / `metadata_error` so they survive the merge in step 3.

In [ ]:
agent_response_df = pd.read_csv(
    os.path.join(run_path, "agent_response.csv"),
).rename(columns={"error": "agent_response_error"})

metadata_df = pd.read_csv(
    os.path.join(run_path, "metadata.csv"),
    converters={
        "generated_cypher": literal_eval,
        "called_tools": literal_eval,
    },
).rename(columns={"error": "metadata_error"})

failed_response_df = pd.read_csv(os.path.join(run_path, "failed_response.csv"))

print(f"Evaluation questions: {len(agent_response_df)}")
print(f"Failed questions:     {len(failed_response_df)}")

## 3. Merge scores with metadata

Join the judge scores from `agent_response.csv` onto the full per-question metadata from
 `metadata.csv`, keyed on `question_id`. This is the same merge the eval's report generator
performs — the resulting `merged` frame drives everything below.

In [ ]:
merged = pd.merge(
    agent_response_df[
        [
            "question_id",
            "rouge_f1_score",
            "factual_correctness_f1_score",
            "answer_relevancy_score",
            "agent_response_error",
        ]
    ],
    metadata_df,
    on="question_id",
)

In [ ]:
merged

## 4. Metric summary

Average the judge scores and the effort metrics across questions.

Two groups of metrics:

- **Judge scores** (`rouge`, `factual_correctness`, `answer_relevancy`) — LLM-judged and
  fairly noisy; treat small differences between runs with caution.
- **Effort metrics** (`num_llm_calls`, `num_tool_calls`, `response_time`) — deterministic
  and directly comparable between runs, e.g. between the `ontology` and `no_ontology`
  conditions.

In [ ]:
score_cols = [
    "rouge_f1_score",
    "factual_correctness_f1_score",
    "answer_relevancy_score",
]
effort_cols = ["num_llm_calls", "num_tool_calls", "response_time"]

merged[score_cols + effort_cols].mean().to_frame("mean")

## 5. Failed questions

Questions that raised an error before the agent could answer. Empty in a healthy run.

In [ ]:
failed_response_df

## 6. Compare expected vs. actual answer for one question

Change the index to inspect a different question.

In [ ]:
i = 0
row = merged.iloc[i]

print(f"Question ID: {row['question_id']}")
print(f"Question:    {row['question']}")
print()
print(f"Expected answer:\n{row['expected_answer']}")
print()
print(f"Agent answer:\n{row['agent_final_answer']}")

## 7. Per-question report

Reuses `create_report_str_from_row` — the same formatter that writes the run's
`report.txt` — so the notebook and the generated report always agree. Each entry shows
the question, the tools the agent called, every generated Cypher query, the judge scores
and the final answer.

The full report is long; collapse this cell's output after skimming, or read
`report.txt` in the run directory instead.

In [ ]:
for _, row in merged.iterrows():
    print(create_report_str_from_row(row))
    print()